# CNN Training Data Extraction Pipeline
Extracts, rotates, and crops data from NPZ files into squares for CNN training.

**Note:** This version has no visualization functions.

In [2]:
import os
import numpy as np
from scipy.ndimage import rotate, zoom, map_coordinates, uniform_filter1d
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import random
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import zipfile


## Utility Functions

In [3]:
def to_float32(array):
    """Convert array to float32 if needed."""
    return array.astype(np.float32) if array.dtype == np.float16 else array


def get_nan_mask(array):
    """Get NaN mask for 2D or 3D arrays."""
    return np.isnan(array[:, :, 0]) if array.ndim == 3 else np.isnan(array)


def get_2d_image(array):
    """Extract 2D image from 2D or 3D array."""
    return array[:, :, 0] if array.ndim == 3 else array


def slice_array(array, y_slice=None, x_slice=None):
    """Slice array handling both 2D and 3D cases."""
    y_slice = y_slice or slice(None)
    x_slice = x_slice or slice(None)
    return array[y_slice, x_slice, :] if array.ndim == 3 else array[y_slice, x_slice]

def normalize_array(array, method='minmax', eps=1e-8):
    """Normalize array values while preserving shape and dtype."""
    # Works on 2D or 3D arrays. Ignores NaNs when computing stats and leaves NaNs in place.
    import numpy as _np
    arr = _np.array(array, copy=True)
    # operate on the 2D image portion if 3D (H,W,C) -> use first channel
    if arr.ndim == 3:
        img = arr[:, :, 0].astype(_np.float32)
    else:
        img = arr.astype(_np.float32)
    mask = _np.isnan(img)
    if method == 'zscore':
        mean = _np.nanmean(img)
        std = _np.nanstd(img)
        std = std if std > 0 else eps
        norm = (img - mean) / std
    else:  # minmax fallback
        vmin = _np.nanmin(img)
        vmax = _np.nanmax(img)
        denom = (vmax - vmin) if (vmax - vmin) > 0 else eps
        norm = (img - vmin) / denom
    # restore NaNs and place back into arr preserving channels
    norm[mask] = _np.nan
    if arr.ndim == 3:
        arr[:, :, 0] = norm.astype(_np.float32)
    else:
        arr = norm.astype(_np.float32)
    return arr

def unzip_file(zip_path, extraction_path):
    """Unzip a file to the specified extraction path."""
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_object:
        zip_object.extractall(extraction_path)

## Configuration

In [4]:
def load_config(config_file=None):
    """Load configuration from file or return defaults."""
    config = {
        'scan_width': None,
        'square_size': 'auto',
        'stride': 0.8,
        'target_size': 128,
    }
    
    if config_file:
        try:
            with open(config_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if '=' in line and not line.startswith('#'):
                        key, value = line.split('=', 1)
                        key, value = key.strip(), value.strip()
                        if key in config:
                            config[key] = int(value) if value.isdigit() else value
        except FileNotFoundError:
            print(f"Config file not found: {config_file}")
    
    return config

## Data Loading

In [5]:
def load_npz_data(npz_path, normalize=True, norm_method='minmax'):
    """Load data from NPZ file. Optionally normalize before returning."""
    data = np.load(npz_path)
    arr = to_float32(data[data.files[0]])
    if normalize:
        return normalize_array(arr, method=norm_method)
    return arr

In [6]:
import pandas as pd

def extract_field(csv_path):
    df = pd.read_csv(csv_path, sep=';')
    df.columns = df.columns.str.strip()

    df["field_file"] = df["field_file"].astype(str).str.strip()

    return (
        df["field_file"],
        df["label"],
        df["width_m"]
    )

## Centerline Detection

In [7]:
def detect_centerline(array, smooth_window=5):
    """
    Detect the centerline of valid data.
    Automatically adapts to vertical or horizontal data orientation.
    
    Returns:
        positions: Array of row/column indices with valid data
        centers: Array of center positions for each row/column
        is_vertical: True if data is more vertical (iterate rows)
    """
    nan_mask = get_nan_mask(array)
    all_valid_y, all_valid_x = np.where(~nan_mask)
    
    if len(all_valid_x) == 0:
        return np.array([]), np.array([]), True
    
    # Determine orientation based on extent
    y_extent = all_valid_y.max() - all_valid_y.min()
    x_extent = all_valid_x.max() - all_valid_x.min()
    is_vertical = y_extent > x_extent
    
    positions, centers = [], []
    
    if is_vertical:
        for y in range(nan_mask.shape[0]):
            valid_x = np.where(~nan_mask[y, :])[0]
            if len(valid_x) > 0:
                positions.append(y)
                centers.append((valid_x[0] + valid_x[-1]) / 2.0)
    else:
        for x in range(nan_mask.shape[1]):
            valid_y = np.where(~nan_mask[:, x])[0]
            if len(valid_y) > 0:
                positions.append(x)
                centers.append((valid_y[0] + valid_y[-1]) / 2.0)
    
    positions, centers = np.array(positions), np.array(centers)
    
    # Smooth centerline
    if smooth_window > 1 and len(centers) > smooth_window:
        centers = uniform_filter1d(centers, size=smooth_window, mode='nearest')
    
    return positions, centers, is_vertical


def analyze_centerline(positions, centers):
    """Analyze centerline for bend detection."""
    if len(positions) < 2:
        return 0.0, 0.0, 0.0
    
    slope, intercept = np.polyfit(positions, centers, 1)
    residuals = centers - (slope * positions + intercept)
    return slope, np.std(residuals), np.degrees(np.arctan(slope))

## Rotation Functions

In [8]:
def estimate_rotation_angle(array):
    """Estimate rotation angle to make diagonal data vertical."""
    arr_2d = get_2d_image(array)
    y_coords, x_coords = np.where(~np.isnan(arr_2d))
    
    if len(x_coords) < 2:
        return 0
    
    slope, _ = np.polyfit(y_coords, x_coords, 1)
    return -np.degrees(np.arctan(slope))


def rotate_to_vertical(image, angle):
    """Rotate image to make data vertical, preserving NaN."""
    image = to_float32(image)
    nan_mask = get_nan_mask(image)
    
    image_temp = np.where(np.isnan(image), 0, image)
    rotated = rotate(image_temp, angle, reshape=True, mode='constant', cval=0)
    rotated_mask = rotate(nan_mask.astype(float), angle, reshape=True, mode='constant', cval=1) > 0.5
    
    if rotated.ndim == 3:
        for c in range(rotated.shape[2]):
            rotated[:, :, c][rotated_mask] = np.nan
    else:
        rotated[rotated_mask] = np.nan
    
    return rotated

## Straightening Functions

In [9]:
def warp_array(array, y_coords_warped, x_coords_warped):
    """Apply coordinate warping to array, preserving NaN regions."""
    is_3d = array.ndim == 3
    
    if is_3d:
        output = np.full_like(array, np.nan)
        for c in range(array.shape[2]):
            channel = np.where(np.isnan(array[:, :, c]), 0, array[:, :, c])
            warped = map_coordinates(channel, [y_coords_warped, x_coords_warped], order=1, mode='constant', cval=0)
            
            valid_mask = map_coordinates((~np.isnan(array[:, :, c])).astype(float), 
                                         [y_coords_warped, x_coords_warped], order=1, mode='constant', cval=0)
            warped[valid_mask < 0.5] = np.nan
            output[:, :, c] = warped
    else:
        temp = np.where(np.isnan(array), 0, array)
        output = map_coordinates(temp, [y_coords_warped, x_coords_warped], order=1, mode='constant', cval=0)
        
        valid_mask = map_coordinates((~np.isnan(array)).astype(float), 
                                     [y_coords_warped, x_coords_warped], order=1, mode='constant', cval=0)
        output[valid_mask < 0.5] = np.nan
    
    return output


def straighten_scan(array, target_center=None):
    """Straighten a bent/curved scan by warping."""
    array = to_float32(array)
    h, w = array.shape[:2]
    
    positions, centers, is_vertical = detect_centerline(array)
    if len(positions) == 0:
        return array
    
    # Create full centerline with interpolation
    if is_vertical:
        full_centers = np.full(h, np.nan)
        full_centers[positions] = centers
        axis_size = h
    else:
        full_centers = np.full(w, np.nan)
        full_centers[positions] = centers
        axis_size = w
    
    valid_mask = ~np.isnan(full_centers)
    if np.sum(valid_mask) > 1:
        full_centers = np.interp(np.arange(axis_size), np.where(valid_mask)[0], full_centers[valid_mask])
    
    if target_center is None:
        target_center = np.mean(centers)
    
    shifts = full_centers - target_center
    y_coords, x_coords = np.mgrid[0:h, 0:w]
    
    if is_vertical:
        x_coords_warped = x_coords + shifts[:, np.newaxis]
        y_coords_warped = y_coords
    else:
        y_coords_warped = y_coords + shifts[np.newaxis, :]
        x_coords_warped = x_coords
    
    return warp_array(array, y_coords_warped, x_coords_warped)


def detect_bend(array, curve_threshold=5.0, verbose=True):
    """Detect if a scan has a significant bend/curve."""
    array = to_float32(array)
    positions, centers, is_vertical = detect_centerline(array)
    slope, curve_std, angle_deg = analyze_centerline(positions, centers)
    has_bend = curve_std >= curve_threshold
    
    if verbose:
        orientation = "vertical" if is_vertical else "horizontal"
        ###print(f"Bend detection ({orientation} data):")
        #print(f"  Linear slope: {slope:.4f} (angle: {angle_deg:.2f} deg)")
        #print(f"  Curve deviation (std): {curve_std:.2f} pixels")
        #print(f"  -> {'BENT scan detected' if has_bend else 'STRAIGHT scan'}")
    
    return {'has_bend': has_bend, 'curve_std': curve_std, 'slope': slope, 
            'angle_deg': angle_deg, 'is_vertical': is_vertical}

## Cropping Functions

In [10]:
def crop_to_width(array, target_width):
    """Crop array to centered strip of specified width."""
    h, w = array.shape[:2]
    nan_mask = get_nan_mask(array)
    _, x_coords = np.where(~nan_mask)
    
    if len(x_coords) == 0:
        return array
    
    center_x = int(np.mean(x_coords))
    half_w = target_width // 2
    x_start = max(0, min(center_x - half_w, w - target_width))
    
    return slice_array(array, x_slice=slice(x_start, x_start + target_width))


def crop_valid_region(array, scan_width=None):
    """Crop to largest valid rectangle (no NaN)."""
    if scan_width is not None:
        cropped = crop_to_width(array, scan_width)
        nan_mask = get_nan_mask(cropped)
        valid_rows = ~np.any(nan_mask, axis=1)
        
        if np.any(valid_rows):
            row_indices = np.where(valid_rows)[0]
            return slice_array(cropped, y_slice=slice(row_indices[0], row_indices[-1] + 1))
        return cropped
    
    # Auto-detect largest rectangle
    nan_mask = get_nan_mask(array)
    h, w = nan_mask.shape
    
    row_ranges = [(np.where(~nan_mask[y, :])[0][[0, -1]] if np.any(~nan_mask[y, :]) else (-1, -1)) 
                  for y in range(h)]
    
    best_area, best_bounds = 0, (0, h, 0, w)
    
    for start_row in range(h):
        if row_ranges[start_row][0] == -1:
            continue
        
        x_min, x_max = row_ranges[start_row]
        
        for end_row in range(start_row, h):
            if row_ranges[end_row][0] == -1:
                break
            
            x_min = max(x_min, row_ranges[end_row][0])
            x_max = min(x_max, row_ranges[end_row][1])
            
            if x_max < x_min:
                break
            
            area = (end_row - start_row + 1) * (x_max - x_min + 1)
            if area > best_area:
                best_area = area
                best_bounds = (start_row, end_row + 1, x_min, x_max + 1)
    
    y_start, y_end, x_start, x_end = best_bounds
    return slice_array(array, y_slice=slice(y_start, y_end), x_slice=slice(x_start, x_end))


def straighten_and_crop(array, scan_width=None):
    """Straighten a bent/curved scan and crop to valid region."""
    array = to_float32(array)
    positions, centers, is_vertical = detect_centerline(array)
    
    if len(positions) < 2:
        #print("Not enough valid data to straighten")
        return array
    
    slope, curve_std, angle_deg = analyze_centerline(positions, centers)
    
    orientation = "vertical" if is_vertical else "horizontal"
    #print(f"Centerline analysis ({orientation} data):")
    #print(f"  Linear slope: {slope:.4f} (angle: {angle_deg:.2f} deg)")
    #print(f"  Curve deviation (std): {curve_std:.2f} pixels")
    
    # Choose method based on curve detection
    if curve_std < 5.0 and abs(slope) > 0.01:
        #print("  -> Using rotation (mostly linear)")
        straightened = rotate_to_vertical(array, estimate_rotation_angle(array))
    else:
        #print("  -> Using adaptive straightening (curved scan)")
        straightened = straighten_scan(array)
    
    # Transpose horizontal data to vertical
    if not is_vertical:
        #print("  Transposing horizontal data to vertical orientation")
        straightened = np.transpose(straightened, (1, 0, 2)) if straightened.ndim == 3 else straightened.T
    
    cropped = crop_valid_region(straightened, scan_width=scan_width)
    
    return cropped

## Square Extraction & Resize

In [11]:
def extract_squares(array, square_size=None, stride=None):
    """Extract fixed-size squares from array (no NaN patches)."""
    h, w = array.shape[:2]
    
    actual_size = w
    
    
    # Normalize stride to integer pixel step
    if stride is None:
        stride_pixels = max(1, actual_size // 2)
    else:
        try:
            s = float(stride)
            if 0.0 < s < 1.0:
                stride_pixels = max(1, int(round(s * actual_size)))
            else:
                stride_pixels = max(1, int(round(s)))
        except Exception:
            stride_pixels = max(1, actual_size // 2)
    
    squares, positions = [], []
    for y in range(0, h - actual_size + 1, stride_pixels):
        for x in range(0, w - actual_size + 1, stride_pixels):
            patch = slice_array(array, slice(y, y + actual_size), slice(x, x + actual_size))
            if not np.any(np.isnan(patch)):
                squares.append(patch)
                positions.append((y, x))
    
    return squares, positions, actual_size


def resize_squares(squares, target_size):
    """Resize all squares to target_size x target_size."""
    if not squares:
        return np.array([])
    
    zoom_factor = target_size / squares[0].shape[0]
    zoom_args = lambda sq: (zoom_factor, zoom_factor, 1) if sq.ndim == 3 else zoom_factor
    
    return np.array([zoom(sq, zoom_args(sq), order=1) for sq in squares])

## Save Functions

In [12]:
def save_squares(resized_squares, original_path, output_dir):
    """Save each resized square as a separate .npy file."""
    if len(resized_squares) == 0:
        #print("No squares to save")
        return []
    
    base_name = os.path.splitext(os.path.basename(original_path))[0]
    
    os.makedirs(output_dir, exist_ok=True)
    
    saved_files = []
    for i, square in enumerate(resized_squares):
        filepath = os.path.join(output_dir, f"{base_name}_square_{i+1:03d}.npz")
        np.savez_compressed(filepath, square)
        saved_files.append(filepath)
    
    #print(f"Saved {len(saved_files)} squares to {output_dir}")
    return saved_files

## Main Pipeline

In [13]:
def process_npz_file(npz_path, config_file=None, scan_width=None, 
                     save=False, output_dir=None, straighten='auto', curve_threshold=5.0):
    """
    Full pipeline to process an NPZ file.
    
    Parameters:
        npz_path: Path to NPZ file
        config_file: Path to config file (optional)
        scan_width: Override scan width (optional)
        save: Whether to save individual squares
        output_dir: Directory to save squares
        straighten: 'auto', True (force), or False (force rotation)
        curve_threshold: Threshold for bend detection in pixels
    
    Returns:
        Resized squares ready for CNN training
    """
    config = load_config(config_file)
    if scan_width is not None:
        config['scan_width'] = scan_width
    
    #print(f"Config: {config}")
    
    array = load_npz_data(npz_path)
    #print(f"Loaded: {array.shape}")
    
    # Determine processing mode
    if straighten == 'auto':
        bend_info = detect_bend(array, curve_threshold=curve_threshold)
        use_straightening = bend_info['has_bend']
        #print(f"Auto-detected mode: {'STRAIGHTENING' if use_straightening else 'ROTATION'}")
    else:
        use_straightening = straighten
    
    # Process based on mode
    if use_straightening:
        #print("Using straightening mode for bent scan...")
        valid = straighten_and_crop(array, scan_width=config['scan_width'])
        angle = 0
    else:
        angle = estimate_rotation_angle(array)
        #print(f"Rotation angle: {angle:.2f} deg")
        rotated = rotate_to_vertical(array, angle)
        #print(f"Rotated shape: {rotated.shape}")
        valid = crop_valid_region(rotated, scan_width=config['scan_width'])
    
    #print(f"Valid region: {valid.shape}")
    
    # Extract squares
    square_size = config['square_size'] if config['square_size'] != 'auto' else None
    squares, positions, actual_size = extract_squares(valid, square_size=square_size, stride=config['stride'])
    #print(f"Extracted {len(squares)} squares (size: {actual_size}x{actual_size})")
    
    # Resize
    resized = resize_squares(squares, config['target_size'])
    #print(f"Resized shape: {resized.shape}")
    
    # Save
    if save and len(resized) > 0:
        save_squares(resized, npz_path, output_dir)
    
    return resized

## Run Pipeline

In [14]:
#paths
ZIP_FILE_PATH = Path("Training_database_float16.zip")
UNZIPPED_PATH = Path("Training_database_float16/Training_database_float16")
CSV_PATH = Path("pipe_detection_label.csv")
CROPPED_ARRAY_OUTPUT_PATH = Path("training_cropped_arrays/")

In [15]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
import os
import shutil
import random



# Read metadata from CSV (same as before)
problem_filenames, solutions, canal_widths = extract_field(CSV_PATH)
file_metadata = dict(zip(problem_filenames, zip(solutions, canal_widths)))
problem_filenames = set(problem_filenames)

# Helper: create a train/test split by copying files into subfolders under the unzipped path.
def create_train_test_split(unzipped_path, train_subdir='train', test_subdir='test', split_ratio=0.8, seed=42, filter_set=None, move=False):
    """Create train/test split directories and copy (or move) matching .npz files."""
    unzipped_path = Path(unzipped_path)
    train_dir = unzipped_path / train_subdir
    test_dir = unzipped_path / test_subdir
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # list all candidate .npz files (filter if a set provided)
    all_files = [f for f in os.listdir(unzipped_path) if f.endswith('.npz')]
    if filter_set is not None:
        all_files = [f for f in all_files if f in filter_set]

    random.seed(seed)
    random.shuffle(all_files)

    n_train = int(len(all_files) * split_ratio)
    train_files = all_files[:n_train]
    test_files = all_files[n_train:]

    # Copy (or move) files into their respective subdirectories. Return relative paths (subdir/filename)
    rel_train = []
    rel_test = []
    for f in train_files:
        src = unzipped_path / f
        dst = train_dir / f
        if not dst.exists():
            if move:
                shutil.move(str(src), str(dst))
            else:
                shutil.copy2(str(src), str(dst))
        rel_train.append(os.path.join(train_subdir, f))
    for f in test_files:
        src = unzipped_path / f
        dst = test_dir / f
        if not dst.exists():
            if move:
                shutil.move(str(src), str(dst))
            else:
                shutil.copy2(str(src), str(dst))
        rel_test.append(os.path.join(test_subdir, f))

    return rel_train, rel_test

# Build the file lists and perform the split (non-destructive copy by default)
all_candidate_files = [f for f in os.listdir(UNZIPPED_PATH) if f.endswith('.npz') and f in problem_filenames]
train_files, test_files = create_train_test_split(UNZIPPED_PATH, train_subdir='train', test_subdir='test', split_ratio=0.8, seed=123, filter_set=problem_filenames, move=False)

# Choose which split to process: 'train', 'test', or 'both'
PROCESS_SPLIT = 'train'  # change to 'test' or 'both' as needed
if PROCESS_SPLIT == 'train':
    files = train_files
elif PROCESS_SPLIT == 'test':
    files = test_files
else:
    files = train_files + test_files

import math
max_workers = min(8, (os.cpu_count() or 4))  # tune this (threads)
chunksize = max(1, math.ceil(len(files) / (max_workers * 4)))

def worker(file):
    # `file` may include a subdirectory (e.g., 'train/xxx.npz')
    # extract basename to look up metadata
    basename = os.path.basename(file)
    solution, canal_width = file_metadata[basename]
    path = os.path.join(UNZIPPED_PATH, file)
    output_path = os.path.join(CROPPED_ARRAY_OUTPUT_PATH, str(solution))
    try:
        return process_npz_file(path, output_dir=output_path, save=True)
    except Exception as e:
        return {"error": str(e), "file": file}

results = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(worker, f): f for f in files}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        results.append(fut.result())

100%|██████████| 2266/2266 [24:29<00:00,  1.54it/s] 

